# PySpark Data Pipeline: Data-Driven Decision Making

**Course:** Big Data & Cloud Computing Technologies  
**Project:** Data-Driven Decision Making in an Organization Using Big Data Technologies  
**Dataset:** 123,847 records × 15 columns (2024–2025)  

---

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| 1. Setup | SparkSession initialization |
| 2. Load | CSV ingestion with explicit schema |
| 3. Profile | Data profiling and null analysis |
| 4. Clean | Duplicates, nulls, outliers, negative values |
| 5. Transform | Feature engineering (9 new features) |
| 6. Aggregate | Department, Region, Quarterly analysis |
| 7. Filter | High error rate & attrition risk cases |
| 8. Save | Parquet output partitioned by Department |

## Stage 1: SparkSession Setup

Initialize Spark with 4GB driver memory and adaptive query execution (AQE) enabled for runtime optimization.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, TimestampType, StringType, DoubleType
)
from pyspark.sql.functions import (
    col, when, count, isnan, isnull, mean as spark_mean, median as spark_median,
    abs as spark_abs, month, quarter, dayofweek, year,
    round as spark_round, lit, desc, avg, countDistinct
)
import os

spark = SparkSession.builder \
    .appName("DataDrivenDecisionPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")

## Stage 2: Define Schema & Load Dataset

We define an explicit schema using `StructType` rather than relying on schema inference. This ensures:
- **Type safety** — catches data type mismatches early
- **Performance** — avoids the extra pass Spark makes to infer types
- **Reproducibility** — schema is documented in code

In [ ]:
schema = StructType([
    StructField("Timestamp",             TimestampType(), True),
    StructField("Employee_ID",           StringType(),    True),
    StructField("Department",            StringType(),    True),
    StructField("Region",                StringType(),    True),
    StructField("Experience_Years",      DoubleType(),    True),
    StructField("Training_Hours",        DoubleType(),    True),
    StructField("Performance_Score",     DoubleType(),    True),
    StructField("Monthly_Sales",         DoubleType(),    True),
    StructField("Customer_Satisfaction", DoubleType(),    True),
    StructField("Attrition_Risk",        DoubleType(),    True),
    StructField("Decision_Type",         StringType(),    True),
    StructField("Data_Quality_Score",    DoubleType(),    True),
    StructField("Processing_Time_sec",   DoubleType(),    True),
    StructField("Error_Rate",            DoubleType(),    True),
    StructField("Decision_Impact_Score", DoubleType(),    True),
])

DATA_PATH = "data_driven_decision_realistic.csv"
OUTPUT_PATH = "processed_data"

print("Loading dataset with explicit schema...")
df = spark.read.csv(DATA_PATH, header=True, schema=schema,
                    timestampFormat="yyyy-MM-dd HH:mm:ss")

print(f"Rows loaded: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

## Stage 3: Data Profiling

Before cleaning, we profile the data to understand:
- How many rows have null/NaN values per column
- Basic statistics (mean, std, min, max) for numeric columns
- Data distribution characteristics

In [ ]:
total_rows = df.count()
print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nUnique Employees: {df.select('Employee_ID').distinct().count():,}")
print(f"Departments: {[r[0] for r in df.select('Department').distinct().collect()]}")
print(f"Regions: {[r[0] for r in df.select('Region').distinct().collect()]}")

In [ ]:
print("\n--- NULL COUNTS PER COLUMN ---")
null_counts = df.select([
    count(when(isnull(c) | isnan(c), c)).alias(c)
    for c in df.columns if df.schema[c].dataType != StringType()
] + [
    count(when(isnull(c), c)).alias(c)
    for c in df.columns if df.schema[c].dataType == StringType()
])
null_counts.show(truncate=False)

In [ ]:
print("--- DESCRIPTIVE STATISTICS ---")
df.describe().show()

## Stage 4: Data Cleaning

Four cleaning operations:

1. **Duplicate Removal** — `dropDuplicates()` removes exact duplicate rows
2. **Negative Value Fix** — `Experience_Years < 0` corrected with `abs()`
3. **Null Imputation** — Column medians for numeric nulls (robust to outliers)
4. **Outlier Capping** — IQR method: values outside [Q1−1.5×IQR, Q3+1.5×IQR] are capped

In [ ]:
# 4.1 Remove exact duplicates
before_dedup = df.count()
df = df.dropDuplicates()
after_dedup = df.count()
print(f"Duplicates removed: {before_dedup - after_dedup}")

In [ ]:
# 4.2 Fix negative Experience_Years
neg_count = df.filter(col("Experience_Years") < 0).count()
df = df.withColumn("Experience_Years",
                    when(col("Experience_Years") < 0, spark_abs(col("Experience_Years")))
                    .otherwise(col("Experience_Years")))
print(f"Fixed {neg_count} negative experience values")

In [ ]:
# 4.3 Impute NULLs with column medians
numeric_cols_to_impute = [
    "Training_Hours", "Monthly_Sales", "Customer_Satisfaction",
    "Data_Quality_Score", "Processing_Time_sec", "Error_Rate"
]

for c in numeric_cols_to_impute:
    median_val = df.select(spark_median(col(c))).collect()[0][0]
    null_count = df.filter(isnull(col(c))).count()
    if median_val is not None:
        df = df.withColumn(c, when(isnull(col(c)), lit(median_val)).otherwise(col(c)))
        print(f"  {c}: filled {null_count:,} NULLs with median = {median_val:.2f}")

In [ ]:
# 4.4 Cap outliers using IQR method
outlier_cols = ["Monthly_Sales", "Processing_Time_sec"]

for c in outlier_cols:
    quantiles = df.approxQuantile(c, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_count = df.filter((col(c) < lower) | (col(c) > upper)).count()
    df = df.withColumn(c,
                        when(col(c) < lower, lit(lower))
                        .when(col(c) > upper, lit(upper))
                        .otherwise(col(c)))
    print(f"  {c}: capped {outlier_count:,} outliers (bounds: [{lower:.2f}, {upper:.2f}])")

print(f"\nCleaned dataset rows: {df.count():,}")

## Stage 5: Feature Engineering

Create 9 new derived features to enhance analytical power:

| Feature | Formula | Purpose |
|---------|---------|--------|
| Experience_Training_Ratio | Experience / (Training + 1) | Training efficiency vs tenure |
| Efficiency_Score | Performance / (Processing_Time + 1) | Output per unit effort |
| Quality_Error_Interaction | Data_Quality × (1 − Error_Rate) | Combined quality-accuracy |
| Month, Quarter, Day_of_Week, Year | Extracted from Timestamp | Temporal analysis |
| Performance_Category | High/Medium/Low binning | Segmentation |
| Risk_Level | Critical/Moderate/Low | Attrition segmentation |

In [ ]:
# Ratio & interaction features
df = df.withColumn("Experience_Training_Ratio",
                    spark_round(col("Experience_Years") / (col("Training_Hours") + 1), 4))

df = df.withColumn("Efficiency_Score",
                    spark_round(col("Performance_Score") / (col("Processing_Time_sec") + 1), 4))

df = df.withColumn("Quality_Error_Interaction",
                    spark_round(col("Data_Quality_Score") * (1 - col("Error_Rate")), 2))

# Temporal features
df = df.withColumn("Month", month(col("Timestamp")))
df = df.withColumn("Quarter", quarter(col("Timestamp")))
df = df.withColumn("Day_of_Week", dayofweek(col("Timestamp")))
df = df.withColumn("Year", year(col("Timestamp")))

# Categorical binning
df = df.withColumn("Performance_Category",
                    when(col("Performance_Score") >= 7.5, "High")
                    .when(col("Performance_Score") >= 4.5, "Medium")
                    .otherwise("Low"))

df = df.withColumn("Risk_Level",
                    when(col("Attrition_Risk") >= 0.7, "Critical")
                    .when(col("Attrition_Risk") >= 0.4, "Moderate")
                    .otherwise("Low"))

print(f"Columns after feature engineering: {len(df.columns)}")
print(f"New features: Experience_Training_Ratio, Efficiency_Score, Quality_Error_Interaction,")
print(f"             Month, Quarter, Day_of_Week, Year, Performance_Category, Risk_Level")
df.select("Experience_Training_Ratio", "Efficiency_Score", "Quality_Error_Interaction",
          "Month", "Quarter", "Performance_Category", "Risk_Level").show(5)

## Stage 6: Aggregations

Three analytical views:
1. **Department-wise** — avg Decision Impact, Performance, Sales, Data Quality, Error Rate
2. **Region-wise** — avg Satisfaction, Attrition Risk, Decision Impact
3. **Quarterly** — temporal trends of Impact, Performance, Error Rate

In [ ]:
print("=== DEPARTMENT-WISE ANALYSIS ===")
dept_agg = df.groupBy("Department").agg(
    spark_round(avg("Decision_Impact_Score"), 2).alias("Avg_Decision_Impact"),
    spark_round(avg("Performance_Score"), 2).alias("Avg_Performance"),
    spark_round(avg("Monthly_Sales"), 2).alias("Avg_Sales"),
    spark_round(avg("Data_Quality_Score"), 2).alias("Avg_Data_Quality"),
    spark_round(avg("Error_Rate"), 4).alias("Avg_Error_Rate"),
    countDistinct("Employee_ID").alias("Unique_Employees"),
    count("*").alias("Record_Count")
).orderBy(desc("Avg_Decision_Impact"))
dept_agg.show(truncate=False)

In [ ]:
print("=== REGION-WISE ANALYSIS ===")
region_agg = df.groupBy("Region").agg(
    spark_round(avg("Customer_Satisfaction"), 2).alias("Avg_Satisfaction"),
    spark_round(avg("Attrition_Risk"), 3).alias("Avg_Attrition_Risk"),
    spark_round(avg("Decision_Impact_Score"), 2).alias("Avg_Decision_Impact"),
    count("*").alias("Record_Count")
).orderBy(desc("Avg_Decision_Impact"))
region_agg.show(truncate=False)

In [ ]:
print("=== QUARTERLY TREND ANALYSIS ===")
quarterly_agg = df.groupBy("Year", "Quarter").agg(
    spark_round(avg("Decision_Impact_Score"), 2).alias("Avg_Impact"),
    spark_round(avg("Performance_Score"), 2).alias("Avg_Performance"),
    spark_round(avg("Error_Rate"), 4).alias("Avg_Error_Rate"),
    count("*").alias("Record_Count")
).orderBy("Year", "Quarter")
quarterly_agg.show(truncate=False)

## Stage 7: Filtering Critical Subsets

Isolate two high-priority subsets for targeted analysis:
1. **High Error Rate** (Error_Rate > 0.3) — records where processing accuracy is below acceptable thresholds
2. **High Attrition Risk** (Attrition_Risk ≥ 0.7) — employees at risk of leaving

In [ ]:
print("=== HIGH ERROR RATE CASES (Error_Rate > 0.3) ===")
high_error = df.filter(col("Error_Rate") > 0.3)
print(f"Records with high error rate: {high_error.count():,} ({high_error.count()/df.count()*100:.1f}%)")

high_error_dept = high_error.groupBy("Department").agg(
    count("*").alias("High_Error_Count"),
    spark_round(avg("Decision_Impact_Score"), 2).alias("Avg_Impact_When_High_Error")
).orderBy(desc("High_Error_Count"))
high_error_dept.show(truncate=False)

In [ ]:
print("=== HIGH ATTRITION RISK EMPLOYEES (Risk >= 0.7) ===")
high_risk = df.filter(col("Attrition_Risk") >= 0.7)
print(f"High-risk records: {high_risk.count():,} ({high_risk.count()/df.count()*100:.1f}%)")

high_risk_summary = high_risk.groupBy("Department").agg(
    countDistinct("Employee_ID").alias("At_Risk_Employees"),
    spark_round(avg("Performance_Score"), 2).alias("Avg_Performance"),
    spark_round(avg("Customer_Satisfaction"), 2).alias("Avg_Satisfaction")
).orderBy(desc("At_Risk_Employees"))
high_risk_summary.show(truncate=False)

## Stage 8: Save Processed Data to Parquet

Save the cleaned and transformed DataFrame to **Apache Parquet** format:
- **Columnar storage** — efficient for analytical queries
- **Snappy compression** — 60–80% size reduction
- **Partitioned by Department** — enables partition pruning during reads

In [ ]:
print(f"Writing processed data to Parquet (partitioned by Department)...")
df.write.mode("overwrite") \
    .partitionBy("Department") \
    .parquet(OUTPUT_PATH)
print(f"Saved to: {OUTPUT_PATH}/")

# Verification
verification = spark.read.parquet(OUTPUT_PATH)
print(f"\nVerification:")
print(f"  Rows in Parquet: {verification.count():,}")
print(f"  Columns: {len(verification.columns)}")
print(f"  Partitions: {[f.name for f in os.scandir(OUTPUT_PATH) if f.is_dir() and f.name.startswith('Department')]}")

## Pipeline Summary

| Metric | Value |
|--------|-------|
| Input rows | 123,847 |
| Input columns | 15 |
| Duplicates removed | ~15 |
| NULLs imputed | 6 columns |
| Outliers capped | 2 columns |
| Features added | 9 new columns |
| Output columns | 24 |
| Output format | Parquet (partitioned by Department) |

In [ ]:
print("=" * 60)
print("  PIPELINE COMPLETE")
print("=" * 60)
print(f"  Input:    {total_rows:,} rows × 15 columns")
print(f"  Output:   {df.count():,} rows × {len(df.columns)} columns")
print(f"  Format:   Parquet, partitioned by Department")
print(f"  Location: {OUTPUT_PATH}/")
print("=" * 60)

spark.stop()
print("\nSpark session stopped.")